In [1]:
import json
import os
import re
import shutil
import warnings
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")


In [2]:

# =========================
# SETTINGS: финальная ML-версия
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42

# Основной метод проекта: Gradient Boosting Regressor.
# Итоговая карта строится из gb_score, без ручного сложения geo_score + bonuses.
GB_MAX_ITER = 450
GB_LEARNING_RATE = 0.025
GB_MAX_LEAF_NODES = 10
GB_MIN_SAMPLES_LEAF = 35
GB_L2_REGULARIZATION = 1.2
GB_VALIDATION_FRACTION = 0.15

# Мягкая целевая переменная: y = exp(-distance_to_evidence / TARGET_SCALE_M).
# Чем меньше TARGET_SCALE_M, тем более локальными будут перспективные зоны.
TARGET_SCALE_M = 2500

# Сглаживание результата на регулярной сетке.
# Увеличь до 3-4, если карта слишком "шумная"; уменьши до 1, если слишком размытая.
SMOOTH_PASSES = 3

# Квантили для перевода расстояний в proximity-признаки.
Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False


In [3]:

# =========================
# PATHS
# =========================
def ensure_input_data() -> Path:
    """Находит папку проекта. Если есть только Прогноз.zip, распаковывает его."""
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base

    zip_candidates = [Path("/mnt/data/Прогноз.zip"), Path.cwd() / "Прогноз.zip"]
    for zp in zip_candidates:
        if zp.exists():
            out = Path("/mnt/data/prog_zip") if str(zp).startswith("/mnt/data") else Path.cwd() / "prog_zip"
            out.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(out)
            if (out / "shp_dbf" / "svita_new.shp").exists():
                return out

    raise FileNotFoundError("Не найден каталог shp_dbf и не найден архив Прогноз.zip.")

BASE_DIR = ensure_input_data()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "gb_final_ml_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_gb_final_ml.gpkg"
OUT_PNG = OUT_DIR / "forecast_gb_final_ml.png"
OUT_COMPARE = OUT_DIR / "compare_gb_final_ml.png"
OUT_PROX = OUT_DIR / "prox_magm_gb_final_ml.png"
OUT_CSV = OUT_DIR / "grid_gb_final_ml.csv"
OUT_JSON = OUT_DIR / "metrics_gb_final_ml.json"

print(f"BASE_DIR: {BASE_DIR}")
print(f"OUT_DIR: {OUT_DIR}")


BASE_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз
OUT_DIR: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result


In [4]:

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    """Select compact high-prospectivity areas for visual highlighting.

    This final ML version does not use old hybrid helper columns such as
    local_bonus or coincidence_score. Yellow polygons are only a display layer:
    they are extracted from the Gradient Boosting prospectivity surface.
    """
    # High-score support area from the ML output.
    q_support = float(grid["prospectivity"].quantile(0.90))
    q_core = float(grid["prospectivity"].quantile(0.96))

    # Local maxima help keep yellow zones compact and close to ML peaks.
    local_peak = local_max_mask(grid, "prospectivity", shape)

    support_zone = grid["prospectivity"] >= q_support
    core_zone = (grid["prospectivity"] >= q_core) | (support_zone & local_peak)

    grid["gold_seed"] = core_zone.astype(int)
    grid["gold_zone"] = keep_large_components(grid, "gold_seed", shape, min_cells=4).astype(int)
    return grid

def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("prox_magm")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


In [5]:
# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

In [6]:

# =========================
# GRID + ML FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

# Признаки близости: чем выше, тем ближе ячейка к фактору.
# Преобразования повторяют геологическую логику методички: cbrt/sqrt для расстояний.
grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

# Сырые трансформированные расстояния. Они нужны бустингу для нелинейных зависимостей.
grid["tr_facies"] = robust_normalize_01(np.cbrt(grid["dist_facies"]), 0.02, 0.98)
grid["tr_paleo"] = robust_normalize_01(np.cbrt(grid["dist_paleo"]), 0.02, 0.98)
grid["tr_tect1"] = robust_normalize_01(np.cbrt(grid["dist_tect1"]), 0.02, 0.98)
grid["tr_tect2"] = robust_normalize_01(np.cbrt(grid["dist_tect2"]), 0.02, 0.98)
grid["tr_struct"] = robust_normalize_01(np.sqrt(grid["dist_struct"]), 0.02, 0.98)
grid["tr_magm"] = robust_normalize_01(np.sqrt(grid["dist_magm"]), 0.02, 0.98)

# Нелинейные геологические сочетания — это не ручной итоговый score, а признаки для ML.
grid["tect_min"] = np.maximum(grid["prox_tect1"], grid["prox_tect2"])
grid["tect_mean"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_cross"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["magm_struct"] = np.sqrt(np.clip(grid["prox_magm"] * grid["prox_struct"], 0, 1))
grid["paleo_struct"] = np.sqrt(np.clip(grid["prox_paleo"] * grid["prox_struct"], 0, 1))
grid["tect_magm"] = np.sqrt(np.clip(grid["tect_mean"] * grid["prox_magm"], 0, 1))
grid["tect_struct"] = np.sqrt(np.clip(grid["tect_mean"] * grid["prox_struct"], 0, 1))
grid["facies_paleo"] = np.sqrt(np.clip(grid["prox_facies"] * grid["prox_paleo"], 0, 1))

# Координаты помогают модели учитывать пространственный тренд, но после робастной нормализации.
centroids = grid.geometry.centroid
grid["x_norm"] = robust_normalize_01(centroids.x, 0.02, 0.98)
grid["y_norm"] = robust_normalize_01(centroids.y, 0.02, 0.98)

feature_cols = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2",
    "tr_facies", "tr_paleo", "tr_struct", "tr_magm", "tr_tect1", "tr_tect2",
    "tect_min", "tect_mean", "tect_cross", "magm_struct", "paleo_struct",
    "tect_magm", "tect_struct", "facies_paleo", "x_norm", "y_norm",
]

print(f"Ячеек в сетке: {len(grid)}")
print(f"Признаков для GB: {len(feature_cols)}")


Ячеек в сетке: 15684
Признаков для GB: 22


In [7]:

# =========================
# SOFT TARGET FROM DIRECT EVIDENCE
# =========================
if points is None or len(points) == 0:
    raise ValueError(
        "Не найдены точки/аномалии для обучения. Для ML-версии нужны прямые поисковые признаки: "
        "геохимические аномалии, рудопроявления или пункты минерализации."
    )

# Расстояние от каждой ячейки до ближайшего прямого поискового признака.
grid = add_distance_feature(grid, points, "dist_evidence")

# Мягкая целевая переменная: рядом с известными признаками значение близко к 1,
# дальше плавно убывает. Это лучше, чем грубые классы 0/1.
grid["target_soft"] = np.exp(-grid["dist_evidence"] / TARGET_SCALE_M)
grid["target_soft"] = np.clip(grid["target_soft"], 0, 1)

# Вес обучения: модель сильнее учитывает ближайшие к проявлениям ячейки,
# но не превращается в бинарную классификацию.
grid["sample_weight"] = 0.35 + 0.65 * grid["target_soft"]

print(f"Точек/аномалий для обучения: {len(points)}")
print(grid[["dist_evidence", "target_soft", "sample_weight"]].describe())


Точек/аномалий для обучения: 1408
       dist_evidence   target_soft  sample_weight
count   15684.000000  15684.000000   15684.000000
mean     5095.999018      0.251871       0.513716
std      3463.290943      0.239489       0.155668
min         0.000000      0.000747       0.350485
25%      2382.107169      0.056412       0.386668
50%      4408.217598      0.171480       0.461462
75%      7187.677615      0.385643       0.600668
max     17998.893529      1.000000       1.000000


In [8]:

# =========================
# GRADIENT BOOSTING — ОСНОВА ПРОЕКТА
# =========================
X = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
y = grid["target_soft"].to_numpy()
w = grid["sample_weight"].to_numpy()

# Пространственные блоки для более честной проверки: соседние ячейки попадают в один block_id.
block_size = 8
blocks_x = (grid["col"].to_numpy() // block_size).astype(int)
blocks_y = (grid["row"].to_numpy() // block_size).astype(int)
groups = blocks_y * (blocks_x.max() + 1) + blocks_x

model = HistGradientBoostingRegressor(
    max_iter=GB_MAX_ITER,
    learning_rate=GB_LEARNING_RATE,
    max_leaf_nodes=GB_MAX_LEAF_NODES,
    min_samples_leaf=GB_MIN_SAMPLES_LEAF,
    l2_regularization=GB_L2_REGULARIZATION,
    validation_fraction=GB_VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
)

# Пространственная cross-validation.
cv_metrics = []
unique_groups = np.unique(groups)
if len(unique_groups) >= 5:
    gkf = GroupKFold(n_splits=5)
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
        fold_model = HistGradientBoostingRegressor(
            max_iter=GB_MAX_ITER,
            learning_rate=GB_LEARNING_RATE,
            max_leaf_nodes=GB_MAX_LEAF_NODES,
            min_samples_leaf=GB_MIN_SAMPLES_LEAF,
            l2_regularization=GB_L2_REGULARIZATION,
            validation_fraction=GB_VALIDATION_FRACTION,
            random_state=RANDOM_STATE + fold,
        )
        fold_model.fit(X[train_idx], y[train_idx], sample_weight=w[train_idx])
        pred = np.clip(fold_model.predict(X[test_idx]), 0, 1)
        cv_metrics.append({
            "fold": fold,
            "mae": float(mean_absolute_error(y[test_idx], pred)),
            "r2": float(r2_score(y[test_idx], pred)),
            "target_mean": float(np.mean(y[test_idx])),
            "pred_mean": float(np.mean(pred)),
        })

# Финальное обучение на всей области прогноза.
model.fit(X, y, sample_weight=w)
grid["gb_score_raw"] = np.clip(model.predict(X), 0, 1)
grid["gb_score_sm"] = smooth_on_regular_grid(grid, "gb_score_raw", grid_shape, passes=SMOOTH_PASSES)
grid["prospectivity"] = robust_normalize_01(grid["gb_score_sm"], 0.02, 0.98)

# Для совместимости с методичкой: там меньше prognoz = лучше. Здесь выше prospectivity = лучше.
grid["prognoz"] = 1.0 - grid["prospectivity"]

# Важность факторов через permutation importance.
# На всей выборке это объяснительная, а не строгая валидационная оценка.
try:
    perm = permutation_importance(
        model, X, y,
        n_repeats=8,
        random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error",
        sample_weight=w,
    )
    feature_importance = dict(
        sorted(
            zip(feature_cols, perm.importances_mean),
            key=lambda item: item[1],
            reverse=True,
        )
    )
except Exception as exc:
    feature_importance = {"error": str(exc)}

print("Gradient Boosting обучен. Итоговая карта = сглаженный gb_score, без ручных бонусов.")
print(pd.DataFrame(cv_metrics) if cv_metrics else "CV не выполнена: мало пространственных блоков.")
print("Top feature importance:")
print(pd.Series(feature_importance).head(12))


Gradient Boosting обучен. Итоговая карта = сглаженный gb_score, без ручных бонусов.
   fold       mae        r2  target_mean  pred_mean
0     1  0.100046  0.654681     0.248713   0.271516
1     2  0.092270  0.701750     0.245531   0.268612
2     3  0.101371  0.666998     0.231821   0.258750
3     4  0.104537  0.633862     0.272534   0.256880
4     5  0.117934  0.528925     0.260756   0.281864
Top feature importance:
x_norm          0.077925
y_norm          0.068403
prox_facies     0.038520
paleo_struct    0.020282
tect_struct     0.014576
facies_paleo    0.012873
prox_struct     0.012604
prox_tect2      0.008599
tect_magm       0.008390
prox_paleo      0.007718
prox_tect1      0.005934
tect_mean       0.004205
dtype: float64


In [9]:

# =========================
# POSTPROCESSING FOR MAP ONLY
# =========================
grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

print(grid[["target_soft", "gb_score_raw", "prospectivity", "prognoz", "gold_zone"]].describe())


        target_soft  gb_score_raw  prospectivity       prognoz     gold_zone
count  15684.000000  15684.000000   15684.000000  15684.000000  15684.000000
mean       0.251871      0.268546       0.395109      0.604891      0.039913
std        0.239489      0.199349       0.284151      0.284151      0.195762
min        0.000747      0.000000       0.000000      0.000000      0.000000
25%        0.056412      0.096854       0.143507      0.386728      0.000000
50%        0.171480      0.234317       0.349680      0.650320      0.000000
75%        0.385643      0.412953       0.613272      0.856493      0.000000
max        1.000000      0.804611       1.000000      1.000000      1.000000


In [10]:

# =========================
# SAVE RESULTS
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "method": "HistGradientBoostingRegressor",
    "description": "Final ML version: Gradient Boosting is the main forecast model; final prospectivity is smoothed gb_score only.",
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "target_scale_m": TARGET_SCALE_M,
    "smooth_passes": SMOOTH_PASSES,
    "point_count": int(len(points)) if points is not None else 0,
    "feature_count": len(feature_cols),
    "feature_cols": feature_cols,
    "cv_metrics": cv_metrics,
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "gb_feature_importance": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")


Готово.
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result\forecast_gb_final_ml.png
COMPARE: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result\compare_gb_final_ml.png
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result\forecast_gb_final_ml.gpkg
CSV: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result\grid_gb_final_ml.csv
JSON: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\gb_final_ml_result\metrics_gb_final_ml.json
